# Invest UAE — Signal Detection Pipeline Demo

End-to-end demonstration of the AI-powered investment signal detection pipeline.

This notebook walks through each stage:
1. **Fetch** articles from 18+ MENA/global RSS sources
2. **Embed** and score relevance using sentence-transformers
3. **Classify** signal types (funding, expansion, partnership, etc.)
4. **Extract** entities (companies, locations, funding amounts)
5. **Score** investability and UAE alignment
6. **Rank** and output the final pipeline

In [ ]:
# ── Self-contained setup: all agents inlined, zero external setup ──
# This cell defines EmbeddingAgent, ClassifierAgent, EntityAgent, ScoringAgent
# directly in the notebook namespace. No uploads, no git clone, no pip installs.
# Works in Google Colab out of the box.

import asyncio
import json
import logging
import hashlib
import re
import math
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Dict, List, Optional, Set, Tuple

import numpy as np

logger = logging.getLogger(__name__)

# ════════════════════════════════════════════════════════════════════
# EmbeddingAgent
# ════════════════════════════════════════════════════════════════════

UAE_INVESTMENT_THEMES = [
    "company expanding operations into the United Arab Emirates Dubai Abu Dhabi",
    "startup raises funding Series A B C venture capital investment round",
    "strategic partnership agreement signed between companies in Middle East",
    "new office opening regional headquarters in UAE MENA Gulf GCC",
    "technology company launches product service in Dubai Abu Dhabi",
    "regulatory approval license granted DIFC ADGM VARA financial authority",
    "merger acquisition deal completed in Middle East North Africa region",
    "executive appointment CEO CTO hire leadership change company growth",
    "cleantech renewable energy sustainability project UAE Net Zero 2050",
    "artificial intelligence AI machine learning company expansion",
    "fintech digital payments banking technology MENA region",
    "real estate development construction project UAE infrastructure",
    "logistics supply chain transport shipping ports Jebel Ali",
    "healthcare medical technology biotech pharmaceutical GCC",
    "manufacturing industrial production Make it in Emirates",
]

SECTOR_DESCRIPTIONS = {
    "fintech": "financial technology digital payments banking blockchain cryptocurrency",
    "artificial_intelligence": "artificial intelligence machine learning deep learning AI neural network",
    "cleantech": "clean technology renewable energy solar wind sustainability green hydrogen",
    "healthcare": "healthcare medical biotech pharmaceutical genomics clinical trials",
    "logistics": "logistics supply chain shipping ports freight transportation warehouse",
    "real_estate": "real estate property development construction infrastructure urban planning",
    "ecommerce": "ecommerce online retail marketplace digital commerce platform",
    "manufacturing": "manufacturing industrial production factory assembly advanced materials",
    "energy": "energy oil gas petroleum LNG power generation utilities",
    "tourism": "tourism hospitality travel hotels entertainment leisure",
    "education": "education edtech learning university school training platform",
    "agritech": "agriculture technology farming food production vertical farming",
    "space": "space technology satellite launch aerospace orbital",
    "defense": "defense security military surveillance cybersecurity",
}


class EmbeddingAgent:
    """Manages text embeddings for semantic similarity and relevance scoring."""

    MODEL_NAME = "all-MiniLM-L6-v2"

    def __init__(self):
        self._model = None
        self._theme_embeddings: Optional[np.ndarray] = None
        self._sector_embeddings: Dict[str, np.ndarray] = {}
        self._cache: Dict[str, np.ndarray] = {}
        self._fallback_mode = False

    def _load_model(self):
        if self._model is not None:
            return
        try:
            from sentence_transformers import SentenceTransformer
            self._model = SentenceTransformer(self.MODEL_NAME)
            logger.info("embedding_model_loaded model=%s", self.MODEL_NAME)
        except ImportError:
            logger.warning(
                "sentence-transformers not installed. Using TF-IDF fallback. "
                "Install with: pip install sentence-transformers"
            )
            self._fallback_mode = True
        except Exception as exc:
            logger.warning("embedding_model_load_failed err=%s, using fallback", exc)
            self._fallback_mode = True

    def _encode_texts(self, texts: List[str]) -> np.ndarray:
        if self._fallback_mode or self._model is None:
            return self._hash_encode(texts)
        return self._model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

    def _hash_encode(self, texts: List[str]) -> np.ndarray:
        dim = 384
        embeddings = []
        for text in texts:
            words = text.lower().split()
            vec = np.zeros(dim)
            for w in words:
                h1 = int(hashlib.md5(w.encode()).hexdigest(), 16)
                h2 = int(hashlib.sha1(w.encode()).hexdigest(), 16)
                idx1 = h1 % dim
                idx2 = h2 % dim
                sign = 1.0 if (h1 >> 32) % 2 == 0 else -1.0
                vec[idx1] += sign
                vec[idx2] += 0.5 * sign
            for i in range(len(words) - 1):
                bigram = f"{words[i]}_{words[i+1]}"
                h = int(hashlib.md5(bigram.encode()).hexdigest(), 16)
                idx = h % dim
                vec[idx] += 0.7
            norm = np.linalg.norm(vec)
            if norm > 0:
                vec /= norm
            embeddings.append(vec)
        return np.array(embeddings)

    def encode(self, text: str) -> np.ndarray:
        self._load_model()
        cache_key = hashlib.sha256(text[:500].encode()).hexdigest()[:16]
        if cache_key in self._cache:
            return self._cache[cache_key]
        emb = self._encode_texts([text])[0]
        self._cache[cache_key] = emb
        return emb

    def encode_batch(self, texts: List[str]) -> np.ndarray:
        self._load_model()
        return self._encode_texts(texts)

    def _get_theme_embeddings(self) -> np.ndarray:
        if self._theme_embeddings is None:
            self._load_model()
            self._theme_embeddings = self._encode_texts(UAE_INVESTMENT_THEMES)
        return self._theme_embeddings

    def relevance_score(self, text: str) -> float:
        article_emb = self.encode(text)
        theme_embs = self._get_theme_embeddings()
        similarities = theme_embs @ article_emb
        max_sim = float(np.max(similarities))
        top3_mean = float(np.mean(np.sort(similarities)[-3:]))
        score = 0.6 * max_sim + 0.4 * top3_mean
        return float(np.clip((score - 0.1) / 0.6, 0.0, 1.0))

    def sector_scores(self, text: str) -> Dict[str, float]:
        if not self._sector_embeddings:
            self._load_model()
            for sector, desc in SECTOR_DESCRIPTIONS.items():
                self._sector_embeddings[sector] = self.encode(desc)
        article_emb = self.encode(text)
        scores = {}
        for sector, sector_emb in self._sector_embeddings.items():
            sim = float(article_emb @ sector_emb)
            scores[sector] = float(np.clip(sim, 0.0, 1.0))
        return scores

    def top_sectors(self, text: str, threshold: float = 0.3, max_sectors: int = 3) -> List[str]:
        scores = self.sector_scores(text)
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        result = [s for s, score in ranked[:max_sectors] if score >= threshold]
        if not result:
            result = self._keyword_sector_detect(text, max_sectors)
        return result if result else ["other"]

    def _keyword_sector_detect(self, text: str, max_sectors: int = 3) -> List[str]:
        text_lower = text.lower()
        SECTOR_KEYWORDS = {
            "fintech": ["fintech", "payment", "banking", "bnpl", "neobank", "crypto", "blockchain", "defi", "digital wallet", "remittance"],
            "artificial_intelligence": ["artificial intelligence", " ai ", "machine learning", "deep learning", "neural", "llm", "generative ai", "computer vision", "nlp"],
            "cleantech": ["cleantech", "clean energy", "renewable", "solar", "wind", "hydrogen", "sustainability", "carbon", "net zero", "green energy", "ev ", "electric vehicle"],
            "healthcare": ["healthcare", "health tech", "biotech", "pharma", "medical", "genomics", "clinical", "hospital", "diagnostics", "telemedicine"],
            "logistics": ["logistics", "supply chain", "shipping", "freight", "port", "warehouse", "delivery", "fleet", "cargo", "transportation"],
            "real_estate": ["real estate", "property", "construction", "infrastructure", "housing", "development project", "urban planning", "tower", "building"],
            "ecommerce": ["ecommerce", "e-commerce", "marketplace", "online retail", "online shopping", "digital commerce", "food delivery", "cloud kitchen"],
            "manufacturing": ["manufacturing", "industrial", "factory", "production", "steel", "assembly", "fabrication", "materials"],
            "energy": ["energy", "oil", "gas", "petroleum", "lng", "power", "utility", "fuel", "opec", "adnoc"],
            "tourism": ["tourism", "hospitality", "travel", "hotel", "resort", "entertainment", "leisure", "theme park", "waterworld"],
            "education": ["education", "edtech", "university", "school", "training", "learning platform", "academic"],
            "agritech": ["agritech", "agriculture", "farming", "food tech", "vertical farm", "food security"],
            "space": ["space", "satellite", "aerospace", "orbital", "launch vehicle", "rocket"],
            "defense": ["defense", "defence", "security", "military", "surveillance", "cybersecurity", "cyber"],
        }
        matches = []
        for sector, keywords in SECTOR_KEYWORDS.items():
            count = sum(1 for kw in keywords if kw in text_lower)
            if count > 0:
                matches.append((sector, count))
        matches.sort(key=lambda x: x[1], reverse=True)
        return [s for s, _ in matches[:max_sectors]]

    def deduplicate(self, texts: List[str], threshold: float = 0.92) -> List[int]:
        if len(texts) <= 1:
            return list(range(len(texts)))
        embeddings = self.encode_batch(texts)
        sim_matrix = embeddings @ embeddings.T
        unique_indices = []
        seen = set()
        for i in range(len(texts)):
            if i in seen:
                continue
            unique_indices.append(i)
            for j in range(i + 1, len(texts)):
                if sim_matrix[i, j] >= threshold:
                    seen.add(j)
        return unique_indices

    def semantic_search(self, query: str, corpus: List[str], top_k: int = 10) -> List[Tuple[int, float]]:
        query_emb = self.encode(query)
        corpus_embs = self.encode_batch(corpus)
        similarities = corpus_embs @ query_emb
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(int(idx), float(similarities[idx])) for idx in top_indices]


# ════════════════════════════════════════════════════════════════════
# ClassifierAgent
# ════════════════════════════════════════════════════════════════════

SIGNAL_TYPE_LABELS = {
    "funding": "Company raises investment capital through funding round, venture capital, Series A/B/C/D, seed round, IPO, or receives financial backing",
    "expansion": "Company opens new office, expands into new market or region, establishes regional headquarters, or enters new country",
    "partnership": "Companies form strategic partnership, sign MoU, create joint venture, or establish collaboration agreement",
    "launch": "Company launches new product, service, platform, or unveils new technology or initiative",
    "regulatory": "Company receives regulatory approval, license, permit, or compliance certification from financial authority",
    "hiring": "Company announces major hiring, appoints executive, names new CEO/CTO, or significantly grows team",
    "m_and_a": "Company announces merger, acquisition, buyout, or takeover of another company",
    "executive": "Senior executive appointment, CEO/CTO/CFO change, board member addition, or leadership transition",
}

SIGNAL_STRENGTH_RULES = {
    "high": [
        "series [a-e]", "raises?\\s+\\$?\\d+[mb]", "acquires", "acquisition",
        "ipo", "unicorn", "billion", "regional headquarters", "secures license",
        "vara", "adgm", "difc", "strategic partnership", "joint venture",
        "appoints ceo", "names new ceo", "uae", "dubai", "abu dhabi",
    ],
    "medium": [
        "seed round", "pre-seed", "launches", "expands", "opens office",
        "partnership", "mou", "hires", "appoints", "collaboration",
        "new market", "new product",
    ],
    "low": [
        "plans to", "considering", "exploring", "may", "could",
        "reportedly", "rumored", "sources say",
    ],
}

KEYWORD_PATTERNS: Dict[str, List[str]] = {
    "funding": [
        r"(?:raises?|raised|secures?|secured|closes?|closed|bags?|bagged|lands?|landed)\s+\$?\d",
        r"series\s+[a-e]", r"seed\s+round", r"pre-seed", r"funding\s+round",
        r"valuation", r"led\s+by", r"backed\s+by", r"venture\s+capital",
        r"investment\s+round", r"ipo\b", r"spac\b", r"pre-ipo",
        r"bridge\s+round", r"growth\s+round", r"mega-?round",
        r"\$\d+[mb]n?\s+(?:in\s+)?(?:funding|investment|capital)",
        r"capital\s+raise", r"fundraise", r"fundraising",
        r"term\s+sheet", r"priced\s+round", r"oversubscribed",
    ],
    "expansion": [
        r"(?:expands?|expanding)\s+(?:into|to|in)", r"opens?\s+(?:new\s+)?office",
        r"launches?\s+in\b", r"enters?\s+(?:the\s+)?market",
        r"regional\s+headquarters", r"set\s+up\s+shop", r"expand\s+operations",
        r"regional\s+hub", r"(?:uae|dubai|abu\s*dhabi|saudi|riyadh)\s+office",
        r"mena\s+expansion", r"new\s+(?:branch|location|facility)",
        r"(?:uae|dubai|abu\s*dhabi)\s+(?:debut|entry|launch)",
        r"sets?\s+up\s+(?:in|shop)", r"picks?\s+(?:dubai|abu\s*dhabi|uae)",
        r"chooses?\s+(?:dubai|abu\s*dhabi|uae)", r"arrives?\s+in\s+(?:uae|dubai)",
        r"establishes?\s+presence", r"opens?\s+doors?\s+in",
        r"scales?\s+(?:into|to|in)", r"goes?\s+global",
        r"crosses?\s+borders?", r"regional\s+expansion",
    ],
    "partnership": [
        r"partners?\s+with", r"partnership\s+with", r"signs?\s+mou",
        r"signs?\s+agreement", r"strategic\s+partnership",
        r"joint\s+venture", r"collaboration\s+with", r"teaming\s+up",
        r"team\s+up\s+with", r"joins?\s+forces\s+with", r"allies?\s+with",
        r"inks?\s+(?:deal|pact|agreement)", r"tie-?up\s+with",
        r"collaborates?\s+with", r"signs?\s+pact", r"forges?\s+alliance",
        r"co-?develop", r"memorandum\s+of\s+understanding",
    ],
    "launch": [
        r"launches?", r"unveils?", r"rolls?\s+out", r"introduces?\s+new",
        r"debuts?", r"goes?\s+live", r"new\s+platform", r"new\s+product",
        r"releases?\s+new", r"goes?\s+to\s+market", r"brings?\s+to\s+market",
        r"ships?\s+(?:new|first)", r"unveils?", r"premieres?",
        r"kicks?\s+off", r"inaugurates?", r"opens?\s+to\s+(?:public|users)",
    ],
    "regulatory": [
        r"regulatory\s+approval", r"license\s+to\s+operate",
        r"(?:granted|receives?|obtains?)\s+license", r"secures?\s+license",
        r"\bvara\b", r"\badgm\b", r"\bdifc\b", r"\bsca\b", r"\bdfsa\b", r"\bfsra\b",
        r"compliance\s+(?:approval|certification)",
        r"in-?principle\s+approval", r"category\s+\d\s+license",
        r"regulatory\s+nod", r"sandbox\s+(?:approval|entry|admission)",
        r"central\s+bank\s+(?:approval|license|nod)",
        r"crypto\s+license", r"payments?\s+license", r"banking\s+license",
        r"(?:vara|adgm|difc|dfsa|fsra|sca)\s+(?:license|approval|nod|greenlight)",
        r"greenlight", r"wins?\s+approval", r"nod\s+from\s+regulator",
        r"cleared\s+by\s+regulator", r"regulator\s+approves",
    ],
    "hiring": [
        r"hiring\s+spree", r"hires?\s+for", r"is\s+hiring",
        r"team\s+grows?", r"headcount", r"talent\s+acquisition",
        r"to\s+(?:hire|recruit)\s+\d+", r"plans?\s+to\s+hire",
        r"double\s+(?:its\s+)?(?:headcount|workforce|team)",
        r"triple\s+(?:its\s+)?(?:headcount|workforce|team)",
        r"creating?\s+\d+\s+(?:new\s+)?jobs", r"\d+\s+new\s+(?:roles|jobs|hires)",
        r"recruitment\s+drive", r"workforce\s+expansion", r"mass\s+hiring",
        r"ramp(?:s|ing)?\s+up\s+(?:hiring|team)", r"boost(?:s|ing)?\s+team",
        r"growing\s+team", r"expanding\s+team", r"adds?\s+\d+\s+(?:jobs|roles)",
        r"plans?\s+to\s+add\s+\d+", r"job\s+creation", r"employment\s+drive",
    ],
    "m_and_a": [
        r"acquires?", r"acquisition\s+of", r"acquired\s+by",
        r"to\s+acquire", r"merger\b", r"merges?\s+with",
        r"buys?\s+out", r"takeover", r"bought\s+by", r"bought\s+out",
        r"majority\s+stake", r"minority\s+stake", r"controlling\s+stake",
        r"buys?\s+\d+%\s+stake", r"acquires?\s+\d+%",
        r"completes?\s+(?:deal|takeover|acquisition|merger)",
        r"consolidat(?:es?|ion)\s+with", r"absorbs?",
        r"snaps?\s+up", r"scoops?\s+up", r"picks?\s+up\s+\w+\s+for\s+\$",
        r"swallows?", r"combines?\s+with", r"divestment",
        r"carve-?out", r"spin-?off", r"sells?\s+(?:to|stake)",
    ],
    "executive": [
        r"appoints?\s+(?:new\s+)?(?:ceo|cto|cfo|coo|cmo|chief)",
        r"names?\s+(?:new\s+)?(?:ceo|cto|cfo|coo|cmo|chief|director|head)",
        r"new\s+chief\s+executive", r"appointed\s+chairman",
        r"new\s+chair\b", r"joins?\s+as\s+(?:ceo|cto|cfo|coo|cmo|chief|md|managing\s+director|president|head|vp|vice\s+president|director|general\s+manager|regional\s+director)",
        r"leadership\s+(?:change|transition|reshuffle)",
        r"steps?\s+down\s+as", r"resigns?\s+as\s+(?:ceo|cto|cfo|chairman)",
        r"promoted\s+to", r"elevated\s+to", r"tapped\s+as",
        r"picks?\s+(?:new\s+)?(?:ceo|cto|cfo|chief)",
        r"hires?\s+(?:new\s+)?(?:ceo|cto|cfo|chief|president|md)",
        r"new\s+(?:managing|regional|country)\s+director",
        r"new\s+general\s+manager", r"new\s+head\s+of",
        r"executive\s+appointment", r"board\s+(?:appointment|addition|change)",
        r"(?:ceo|cto|cfo|coo)\s+(?:departs|exits|leaves)",
        r"succeeds?\s+(?:as|in\s+the\s+role)", r"takes?\s+(?:over|the\s+reins)\s+as",
    ],
}


@dataclass
class SignalClassification:
    signal_type: str
    confidence: float
    strength: str
    all_scores: Dict[str, float]


class ClassifierAgent:
    """Classifies articles into investment signal types."""

    def __init__(self, use_ml: bool = True):
        self._pipeline = None
        self._use_ml = use_ml
        self._ml_available = False
        self._embedding_agent = None

    def _load_pipeline(self):
        if self._pipeline is not None or not self._use_ml:
            return
        # Inlined EmbeddingAgent — always available in this namespace
        self._embedding_agent = EmbeddingAgent()  # inlined class
        self._ml_available = True
        logger.info("classifier_using_embeddings")
        return

    def _keyword_classify(self, text: str) -> Dict[str, float]:
        text_lower = text.lower()
        scores: Dict[str, float] = {}
        for signal_type, patterns in KEYWORD_PATTERNS.items():
            match_count = sum(1 for p in patterns if re.search(p, text_lower))
            scores[signal_type] = min(1.0, match_count * 0.4)
        return scores

    def _embedding_classify(self, text: str) -> Dict[str, float]:
        if self._embedding_agent is None:
            return self._keyword_classify(text)
        text_emb = self._embedding_agent.encode(text)
        scores = {}
        for signal_type, description in SIGNAL_TYPE_LABELS.items():
            desc_emb = self._embedding_agent.encode(description)
            similarity = float(text_emb @ desc_emb)
            scores[signal_type] = float(max(0.0, min(1.0, similarity)))
        return scores

    def _zeroshot_classify(self, text: str) -> Dict[str, float]:
        if self._pipeline is None:
            return self._keyword_classify(text)
        labels = list(SIGNAL_TYPE_LABELS.values())
        label_to_type = {v: k for k, v in SIGNAL_TYPE_LABELS.items()}
        result = self._pipeline(text[:512], candidate_labels=labels, multi_label=True)
        scores = {}
        for label, score in zip(result["labels"], result["scores"]):
            signal_type = label_to_type.get(label, "")
            if signal_type:
                scores[signal_type] = float(score)
        return scores

    def _assess_strength(self, text: str, signal_type: str) -> str:
        text_lower = text.lower()
        for pattern in SIGNAL_STRENGTH_RULES["high"]:
            if re.search(pattern, text_lower):
                return "high"
        for pattern in SIGNAL_STRENGTH_RULES["low"]:
            if re.search(pattern, text_lower):
                return "low"
        return "medium"

    def classify(self, text: str) -> SignalClassification:
        self._load_pipeline()
        keyword_scores = self._keyword_classify(text)
        if self._ml_available and self._embedding_agent:
            ml_scores = self._embedding_classify(text)
            scores = {
                k: 0.6 * ml_scores.get(k, 0) + 0.4 * keyword_scores.get(k, 0)
                for k in set(list(ml_scores.keys()) + list(keyword_scores.keys()))
            }
        elif self._ml_available and self._pipeline:
            ml_scores = self._zeroshot_classify(text)
            scores = {
                k: 0.7 * ml_scores.get(k, 0) + 0.3 * keyword_scores.get(k, 0)
                for k in set(list(ml_scores.keys()) + list(keyword_scores.keys()))
            }
        else:
            scores = keyword_scores
        if not scores or max(scores.values()) < 0.05:
            return SignalClassification(signal_type="launch", confidence=0.0, strength="low", all_scores=scores)
        best_type = max(scores, key=scores.get)  # type: ignore
        confidence = scores[best_type]
        strength = self._assess_strength(text, best_type)
        return SignalClassification(signal_type=best_type, confidence=confidence, strength=strength, all_scores=scores)

    def classify_batch(self, texts: List[str]) -> List[SignalClassification]:
        return [self.classify(text) for text in texts]

    def is_investment_signal(self, text: str, threshold: float = 0.15) -> bool:
        result = self.classify(text)
        return result.confidence >= threshold


# ════════════════════════════════════════════════════════════════════
# EntityAgent
# ════════════════════════════════════════════════════════════════════

MENA_CITIES = {
    "dubai": ("United Arab Emirates", "AE", "Dubai", 25.2048, 55.2708),
    "abu dhabi": ("United Arab Emirates", "AE", "Abu Dhabi", 24.4539, 54.3773),
    "sharjah": ("United Arab Emirates", "AE", "Sharjah", 25.3573, 55.4033),
    "ras al khaimah": ("United Arab Emirates", "AE", "Ras Al Khaimah", 25.6741, 55.9804),
    "ajman": ("United Arab Emirates", "AE", "Ajman", 25.4052, 55.5136),
    "riyadh": ("Saudi Arabia", "SA", "Riyadh", 24.7136, 46.6753),
    "jeddah": ("Saudi Arabia", "SA", "Jeddah", 21.4858, 39.1925),
    "neom": ("Saudi Arabia", "SA", "NEOM", 28.0339, 35.0934),
    "dammam": ("Saudi Arabia", "SA", "Dammam", 26.4207, 50.0888),
    "doha": ("Qatar", "QA", "Doha", 25.2854, 51.531),
    "manama": ("Bahrain", "BH", "Manama", 26.2285, 50.586),
    "kuwait city": ("Kuwait", "KW", "Kuwait City", 29.3759, 47.9774),
    "muscat": ("Oman", "OM", "Muscat", 23.6139, 58.5922),
    "cairo": ("Egypt", "EG", "Cairo", 30.0444, 31.2357),
    "amman": ("Jordan", "JO", "Amman", 31.9454, 35.9284),
    "beirut": ("Lebanon", "LB", "Beirut", 33.8938, 35.5018),
    "casablanca": ("Morocco", "MA", "Casablanca", 33.5731, -7.5898),
    "istanbul": ("Turkey", "TR", "Istanbul", 41.008, 28.978),
    "tel aviv": ("Israel", "IL", "Tel Aviv", 32.0853, 34.7818),
}

GLOBAL_CITIES = {
    "san francisco": ("United States", "US", "San Francisco", 37.7749, -122.4194),
    "new york": ("United States", "US", "New York", 40.7128, -74.006),
    "silicon valley": ("United States", "US", "San Francisco", 37.3861, -122.0839),
    "london": ("United Kingdom", "GB", "London", 51.5074, -0.1278),
    "singapore": ("Singapore", "SG", "Singapore", 1.3521, 103.8198),
    "berlin": ("Germany", "DE", "Berlin", 52.52, 13.405),
    "paris": ("France", "FR", "Paris", 48.8566, 2.3522),
    "mumbai": ("India", "IN", "Mumbai", 19.076, 72.8777),
    "bangalore": ("India", "IN", "Bangalore", 12.9716, 77.5946),
    "shanghai": ("China", "CN", "Shanghai", 31.2304, 121.4737),
    "hong kong": ("China", "CN", "Hong Kong", 22.3193, 114.1694),
    "tokyo": ("Japan", "JP", "Tokyo", 35.6762, 139.6503),
    "seoul": ("South Korea", "KR", "Seoul", 37.5665, 126.978),
    "sydney": ("Australia", "AU", "Sydney", -33.8688, 151.2093),
    "zurich": ("Switzerland", "CH", "Zurich", 47.3769, 8.5417),
    "amsterdam": ("Netherlands", "NL", "Amsterdam", 52.3676, 4.9041),
}

ALL_CITIES = {**MENA_CITIES, **GLOBAL_CITIES}

COUNTRY_NAMES = {
    "uae": ("United Arab Emirates", "AE"),
    "united arab emirates": ("United Arab Emirates", "AE"),
    "saudi arabia": ("Saudi Arabia", "SA"),
    "saudi": ("Saudi Arabia", "SA"),
    "ksa": ("Saudi Arabia", "SA"),
    "qatar": ("Qatar", "QA"),
    "bahrain": ("Bahrain", "BH"),
    "kuwait": ("Kuwait", "KW"),
    "oman": ("Oman", "OM"),
    "egypt": ("Egypt", "EG"),
    "jordan": ("Jordan", "JO"),
    "lebanon": ("Lebanon", "LB"),
    "morocco": ("Morocco", "MA"),
    "turkey": ("Turkey", "TR"),
    "israel": ("Israel", "IL"),
    "pakistan": ("Pakistan", "PK"),
    "india": ("India", "IN"),
    "united states": ("United States", "US"),
    "united kingdom": ("United Kingdom", "GB"),
    "uk": ("United Kingdom", "GB"),
    "us": ("United States", "US"),
    "usa": ("United States", "US"),
    "china": ("China", "CN"),
    "japan": ("Japan", "JP"),
    "germany": ("Germany", "DE"),
    "france": ("France", "FR"),
    "singapore": ("Singapore", "SG"),
    "south korea": ("South Korea", "KR"),
    "australia": ("Australia", "AU"),
    "switzerland": ("Switzerland", "CH"),
    "netherlands": ("Netherlands", "NL"),
    "iraq": ("Iraq", "IQ"),
    "tunisia": ("Tunisia", "TN"),
    "algeria": ("Algeria", "DZ"),
}

COMPANY_SUFFIXES = re.compile(
    r"\s*\b(?:Inc\.?|Ltd\.?|LLC|PLC|Corp\.?|Corporation|Company|Co\.?|PJSC|JSC|"
    r"Holdings?|Group|International|Technologies|Tech|Solutions|Ventures?|"
    r"Capital|Partners|Management|Fund|Limited)\b\.?\s*$",
    re.IGNORECASE,
)

FUNDING_PATTERNS = [
    re.compile(r"(?:USD|\$|US\$)\s*([\d,]+\.?\d*)\s*(billion|million|[BMK])\b", re.I),
    re.compile(r"([\d,]+\.?\d*)\s*(billion|million|thousand)\s*(?:US\s+)?dollars?", re.I),
    re.compile(r"(?:raises?|raised|secures?|secured|closes?)\s+(?:USD|\$|US\$)\s*([\d,]+\.?\d*)\s*([BMK]?)", re.I),
]

MULTIPLIERS = {
    "b": 1_000_000_000,
    "billion": 1_000_000_000,
    "m": 1_000_000,
    "million": 1_000_000,
    "k": 1_000,
    "thousand": 1_000,
}


@dataclass
class ExtractedEntity:
    name: str
    aliases: List[str] = field(default_factory=list)
    description: Optional[str] = None
    headquarters_country: Optional[str] = None
    headquarters_country_code: Optional[str] = None
    headquarters_city: Optional[str] = None
    headquarters_lat: Optional[float] = None
    headquarters_lng: Optional[float] = None
    expansion_countries: List[str] = field(default_factory=list)
    expansion_cities: List[str] = field(default_factory=list)
    funding_usd: Optional[float] = None
    executives: List[str] = field(default_factory=list)


class EntityAgent:
    """Extracts structured entities from news article text."""

    def __init__(self):
        self._known_companies: Set[str] = set()

    def extract_funding(self, text: str) -> Optional[float]:
        amounts: List[float] = []
        for pattern in FUNDING_PATTERNS:
            for match in pattern.finditer(text):
                try:
                    num_str = match.group(1).replace(",", "")
                    num = float(num_str)
                    suffix = (match.group(2) or "").lower().strip()
                    multiplier = MULTIPLIERS.get(suffix, 1)
                    if suffix == "" and num < 100:
                        multiplier = 1_000_000
                    amounts.append(num * multiplier)
                except (ValueError, IndexError):
                    continue
        return max(amounts) if amounts else None

    def extract_locations(self, text: str) -> Tuple[List[str], List[str]]:
        text_lower = text.lower()
        cities_found = []
        countries_found = []
        for city_key, info in ALL_CITIES.items():
            if city_key in text_lower:
                cities_found.append(city_key)
        for country_key, info in COUNTRY_NAMES.items():
            if country_key in text_lower:
                countries_found.append(country_key)
        return cities_found, countries_found

    _FRAGMENT_WORDS = {
        "round", "series", "million", "billion", "funding", "investment",
        "investors", "raised", "raises", "secured", "secures", "backed",
        "backs", "led", "participation", "participated", "announced",
        "announces", "the", "this", "these", "those", "their", "its",
        "a", "an", "some", "new", "existing", "among", "including",
        "leads", "boosts", "launches", "signs", "drives", "hosts",
        "forges", "formalises", "formalise", "finalises", "finalise",
        "commits", "targets", "unveils", "plans", "enters",
        "acquires", "completes", "doubles", "triples",
        "expands", "inks", "bags", "lands", "debuts",
        "strategic", "comprehensive",
        "form", "footprint", "port", "legend", "lendtech",
        "sqm", "said", "they", "are",
        "page", "going", "live", "without", "any", "time", "later",
        "week", "year", "month", "day", "ago", "much", "many", "every",
        "most", "less", "more", "next", "last", "good", "better", "best",
        "great", "huge", "quick", "quickly", "fast", "soon",
        "today", "yesterday", "tomorrow", "already", "still", "now",
        "here", "there", "then", "once", "yet", "either",
    }

    _STOPWORD_STARTS = {
        "the", "a", "an", "this", "that", "these", "those", "some",
        "new", "existing", "several", "other", "its", "their",
        "among", "with", "by", "from", "for", "in", "on", "of",
    }

    _COUNTRY_SINGLE_TOKENS = {
        "iran", "israel", "syria", "russia", "ukraine", "yemen",
        "afghanistan", "turkey", "libya", "sudan", "belarus",
        "cuba", "venezuela", "myanmar", "eritrea", "somalia",
        "germany", "france", "italy", "spain", "brazil", "mexico",
        "canada", "australia", "japan", "indonesia", "thailand",
        "vietnam", "philippines", "malaysia", "poland", "netherlands",
        "turkey", "greece", "portugal", "norway", "sweden", "finland",
        "denmark", "ireland", "austria", "switzerland", "belgium",
        "nigeria", "kenya", "ethiopia", "ghana", "senegal",
    }

    _TOPIC_WORDS = {
        "education", "infrastructure", "proptech", "fintech",
        "cleantech", "healthcare", "logistics", "manufacturing",
        "tourism", "defense", "defence", "energy", "agritech",
        "biotech", "edtech", "insurtech", "regtech", "mobility",
        "retail", "ecommerce", "e-commerce", "aviation", "telecom",
        "media", "gaming", "adtech", "martech", "hrtech",
    }

    _PUBLICATION_STARTS = {
        "wired", "techcrunch", "reuters", "bloomberg", "forbes",
        "cnn", "bbc", "guardian", "economist", "axios", "verge",
        "engadget", "gizmodo", "wamda", "magnitt", "menabytes",
        "sifted", "crunchbase", "venturebeat", "zawya", "agbi",
        "khaleej", "gulf", "arabian", "skift",
    }

    _GENERIC_PREFIXES = {
        "ai", "ml", "nlp", "iot", "ar", "vr", "dl",
        "saudi", "uae", "us", "uk", "indian", "chinese", "japanese",
        "african", "european", "asian", "arabian", "gulf", "mena",
        "american", "global", "international", "regional", "national",
    }

    def _looks_like_company(self, name: str) -> bool:
        return EntityAgent._looks_like_company_name(name)

    @classmethod
    def _looks_like_company_name(cls, name: str) -> bool:
        if not name or len(name) < 2:
            return False
        trimmed = name.strip()
        if re.search(r"[.,;:\-]\s*$", trimmed):
            return False
        if re.search(r"\b[A-Z][a-z]+\.[A-Z]", trimmed):
            return False
        words = trimmed.split()
        if not words:
            return False
        if len(words) > 6:
            return False
        if words[0].lower() in cls._STOPWORD_STARTS:
            return False
        if len(words[0]) == 1 and "." not in words[0]:
            return False
        if not any(c.isalpha() for c in trimmed):
            return False
        lowered_tokens = [w.strip(".,;:()'\"").lower() for w in words]
        lowered_set = set(lowered_tokens)
        if lowered_set & cls._FRAGMENT_WORDS:
            return False
        glue = {
            "of", "and", "for", "with", "from", "to", "in", "on", "at",
            "by", "the", "a", "an", "or", "is", "are", "was", "were",
            "said", "that", "which", "who", "whom", "been", "being",
            "have", "has", "had", "they", "their",
        }
        glue_count = sum(1 for t in lowered_tokens if t in glue)
        if glue_count >= 2:
            return False
        if trimmed == trimmed.lower():
            return False
        if trimmed == trimmed.upper() and len(trimmed) > 4:
            return False
        if lowered_tokens[-1] in {
            "the", "a", "an", "and", "or", "of", "by", "for", "to",
            "in", "on", "at", "with", "from",
            "has", "have", "had", "was", "were", "is", "are", "will",
            "would", "been", "being", "also", "not", "does", "did",
            "just", "only", "now", "recently", "still", "then",
            "other", "others",
            "major", "key", "new", "top", "first",
        }:
            return False
        if len(words) == 1 and words[0].lower() in cls._COUNTRY_SINGLE_TOKENS:
            return False
        if len(words) == 1 and words[0].lower() in cls._TOPIC_WORDS:
            return False
        if len(words) >= 2 and all(
            (t in cls._TOPIC_WORDS
             or t in cls._GENERIC_PREFIXES
             or t in cls._COUNTRY_SINGLE_TOKENS)
            for t in lowered_tokens
        ):
            return False
        if words[0].lower() in cls._PUBLICATION_STARTS and len(words) >= 2:
            return False
        if len(words) >= 3 and any(
            t in {"in", "with", "for", "by", "from", "at", "on"}
            for t in lowered_tokens[1:-1]
        ):
            return False
        if len(words) == 1:
            if words[0].lower() in {
                "dubai", "abu", "sharjah", "uae", "ksa", "qatar", "oman",
                "bahrain", "kuwait", "egypt", "jordan", "lebanon", "iraq",
                "morocco", "tunisia", "algeria", "saudi", "china", "india",
                "pakistan", "usa", "us", "uk", "eu", "africa", "asia",
                "europe", "mena", "gcc", "emirates", "jeddah", "riyadh",
                "cairo", "doha", "amman", "beirut", "muscat", "manama",
                "market", "business", "news", "global", "his", "why",
                "what", "how", "when", "where", "who", "which",
            }:
                return False
        if any(re.fullmatch(r"(19|20)\d{2}", t) for t in words):
            return False
        if any(re.fullmatch(r"\d+(?:\.\d+)?[a-z]{0,3}", t.lower()) for t in words if not t.isalpha()):
            return False
        if len(words) >= 3 and words[1].lower() in {"as", "and", "is", "was"}:
            return False
        if len(words) == 1 and len(words[0]) <= 2:
            return False
        return True

    def extract_company_names(self, text: str) -> List[str]:
        companies: List[str] = []
        action_pattern = re.compile(
            r"([A-Z][A-Za-z0-9\s&\-\.]{1,50}?)\s+"
            r"(?:raises?|launches?|expands?|partners?|acquires?|opens?|"
            r"secures?|signs?|announces?|completes?|unveils?|begins?|"
            r"receives?|obtains?|closes?|appoints?|names?|hires?|"
            r"promotes?|taps?|picks?|elevates?|invests?|backs?|"
            r"inks?|rolls?|introduces?|debuts?|bags?|lands?|"
            r"joins?|teams?|forges?|enters?|plans?|doubles?|triples?)",
        )
        for match in action_pattern.finditer(text):
            name = match.group(1).strip()
            name = COMPANY_SUFFIXES.sub("", name).strip()
            if self._looks_like_company(name):
                companies.append(name)
        exec_of_pattern = re.compile(
            r"(?:CEO|CFO|CTO|COO|CMO|Chairman|President|Director|"
            r"Managing\s+Director|General\s+Manager|Head|VP|Vice\s+President)"
            r"\s+(?:of|at|for)\s+([A-Z][A-Za-z0-9\s&\-\.]{2,40}?)(?:\.|,|$|\s+(?:in|to|from|after|said|is|was))",
        )
        for match in exec_of_pattern.finditer(text):
            name = match.group(1).strip()
            name = COMPANY_SUFFIXES.sub("", name).strip()
            if self._looks_like_company(name):
                companies.append(name)
        possessive_pattern = re.compile(
            r"([A-Z][A-Za-z0-9\s&\-\.]{2,40}?)(?:'s|'s)\s+"
            r"(?:new\s+)?(?:CEO|CFO|CTO|COO|Chairman|President|Director|"
            r"Managing\s+Director|General\s+Manager)",
        )
        for match in possessive_pattern.finditer(text):
            name = match.group(1).strip()
            name = COMPANY_SUFFIXES.sub("", name).strip()
            if self._looks_like_company(name):
                companies.append(name)
        indicator_pattern = re.compile(
            r"(?:company|startup|firm|unicorn|fintech|platform)\s+"
            r"([A-Z][A-Za-z0-9\s&\-\.]{2,40})",
        )
        for match in indicator_pattern.finditer(text):
            name = match.group(1).strip()
            name = COMPANY_SUFFIXES.sub("", name).strip()
            if self._looks_like_company(name):
                companies.append(name)
        seen = set()
        unique = []
        for c in companies:
            key = c.lower().strip()
            if key not in seen:
                seen.add(key)
                unique.append(c)
        return unique[:10]

    def extract_executives(self, text: str) -> List[str]:
        executives = []
        exec_pattern = re.compile(
            r"(?:(?:CEO|CTO|CFO|COO|CMO|Chairman|President|Director|"
            r"Chief\s+\w+\s+Officer|Managing\s+Director|General\s+Manager)\s+)"
            r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3})",
        )
        for match in exec_pattern.finditer(text):
            executives.append(match.group(0).strip())
        titled_pattern = re.compile(
            r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,3}),?\s+"
            r"(?:the\s+)?(?:CEO|CTO|CFO|COO|chairman|president|director|founder)"
        )
        for match in titled_pattern.finditer(text):
            executives.append(match.group(0).strip())
        return list(set(executives))[:5]

    def infer_headquarters(self, text: str, cities: List[str], countries: List[str]):
        text_lower = text.lower()
        hq_patterns = [
            r"(?:headquartered|based|hq)\s+in\s+(\w[\w\s]*\w)",
            r"(\w[\w\s]*\w)\s*-?\s*based\s+(?:company|startup|firm)",
        ]
        for pattern in hq_patterns:
            match = re.search(pattern, text_lower)
            if match:
                location = match.group(1).strip()
                if location in ALL_CITIES:
                    info = ALL_CITIES[location]
                    return info[0], info[1], info[2], info[3], info[4]
                if location in COUNTRY_NAMES:
                    info = COUNTRY_NAMES[location]
                    return info[0], info[1], None, None, None
        if cities:
            city = cities[0]
            if city in ALL_CITIES:
                info = ALL_CITIES[city]
                return info[0], info[1], info[2], info[3], info[4]
        if countries:
            country = countries[0]
            if country in COUNTRY_NAMES:
                info = COUNTRY_NAMES[country]
                return info[0], info[1], None, None, None
        return None, None, None, None, None

    def infer_expansion_targets(self, text: str, hq_city: Optional[str], cities: List[str]):
        text_lower = text.lower()
        targets = []
        expansion_patterns = [
            r"(?:expands?|expanding|enters?|entering|launches?\s+in|"
            r"opens?\s+(?:new\s+)?office\s+in|"
            r"(?:new|regional)\s+headquarters?\s+in)\s+"
            r"(\w[\w\s]*\w)",
        ]
        for pattern in expansion_patterns:
            for match in re.finditer(pattern, text_lower):
                location = match.group(1).strip()
                if location in ALL_CITIES:
                    info = ALL_CITIES[location]
                    if hq_city and location == hq_city.lower():
                        continue
                    targets.append((info[0], info[1], info[2]))
                elif location in COUNTRY_NAMES:
                    info = COUNTRY_NAMES[location]
                    targets.append((info[0], info[1], None))
        seen = set()
        unique = []
        for t in targets:
            key = (t[0], t[2])
            if key not in seen:
                seen.add(key)
                unique.append(t)
        return unique[:5]

    def extract(self, title: str, text: str) -> List[ExtractedEntity]:
        full_text = f"{title} {text}"
        companies = self.extract_company_names(full_text)
        cities, countries = self.extract_locations(full_text)
        funding = self.extract_funding(full_text)
        executives = self.extract_executives(full_text)
        hq_country, hq_code, hq_city, hq_lat, hq_lng = self.infer_headquarters(full_text, cities, countries)
        expansion_targets = self.infer_expansion_targets(full_text, hq_city, cities)
        entities = []
        for name in companies:
            entity = ExtractedEntity(
                name=name,
                headquarters_country=hq_country,
                headquarters_country_code=hq_code,
                headquarters_city=hq_city,
                headquarters_lat=hq_lat,
                headquarters_lng=hq_lng,
                expansion_countries=[t[0] for t in expansion_targets],
                expansion_cities=[t[2] for t in expansion_targets if t[2]],
                funding_usd=funding,
                executives=executives,
            )
            entities.append(entity)
        return entities


# ════════════════════════════════════════════════════════════════════
# ScoringAgent
# ════════════════════════════════════════════════════════════════════

SECTOR_UAE_WEIGHTS: Dict[str, float] = {
    "artificial_intelligence": 0.95,
    "cleantech": 0.92,
    "fintech": 0.90,
    "energy": 0.88,
    "manufacturing": 0.87,
    "healthcare": 0.85,
    "space": 0.85,
    "defense": 0.83,
    "logistics": 0.82,
    "education": 0.75,
    "real_estate": 0.72,
    "ecommerce": 0.70,
    "tourism": 0.68,
    "agritech": 0.65,
    "other": 0.50,
}

SIGNAL_INVESTABILITY_WEIGHTS: Dict[str, float] = {
    "funding": 15.0, "expansion": 12.0, "m_and_a": 14.0, "partnership": 10.0,
    "regulatory": 11.0, "launch": 8.0, "hiring": 6.0, "executive": 5.0,
}

STRENGTH_MULTIPLIERS: Dict[str, float] = {"high": 1.5, "medium": 1.0, "low": 0.6}

UAE_COUNTRY_CODE = "AE"
MENA_COUNTRY_CODES = {"AE", "SA", "QA", "BH", "KW", "OM", "EG", "JO", "LB", "MA", "TN", "IQ"}
GCC_COUNTRY_CODES = {"AE", "SA", "QA", "BH", "KW", "OM"}

UAE_STRATEGY_TEXTS = [
    "UAE National AI Strategy 2031 artificial intelligence innovation",
    "UAE Net Zero 2050 clean energy sustainability carbon neutral",
    "Operation 300 billion industrial GDP manufacturing",
    "Make it in the Emirates industrial incentives local production",
    "DIFC ADGM VARA fintech regulation financial innovation hub",
    "Abu Dhabi Mubadala ADIA ADQ sovereign wealth fund investment",
    "Dubai economic diversification global business hub",
    "Golden Visa Green Visa talent attraction immigration",
    "Etihad Rail infrastructure mega projects logistics",
    "Food security agriculture technology water sustainability",
]


@dataclass
class CompanyScores:
    investability_score: float
    uae_alignment_score: float
    composite_score: float
    investability_breakdown: Dict[str, float]
    alignment_breakdown: Dict[str, float]
    thesis_snippet: str


class ScoringAgent:
    """Scores companies for investability and UAE alignment."""

    def __init__(self):
        self._embedding_agent = None
        self._strategy_embeddings = None

    def _try_load_embeddings(self):
        if self._embedding_agent is not None:
            return
        # Inlined EmbeddingAgent — always available in this namespace
        self._embedding_agent = EmbeddingAgent()  # inlined class
        logger.info("scoring_agent_embeddings_loaded")

    def _score_investability(self, signals, sectors, funding_usd, description):
        breakdown = {}
        signal_score = 0.0
        for sig in signals:
            sig_type = sig.get("type", "launch")
            sig_strength = sig.get("strength", "medium")
            weight = SIGNAL_INVESTABILITY_WEIGHTS.get(sig_type, 5.0)
            multiplier = STRENGTH_MULTIPLIERS.get(sig_strength, 1.0)
            signal_score += weight * multiplier
        signal_factor = min(35.0, 10.0 * math.log1p(signal_score / 10.0))
        breakdown["signal_momentum"] = round(signal_factor, 1)

        funding_factor = 0.0
        if funding_usd:
            if funding_usd >= 1_000_000_000:
                funding_factor = 25.0
            elif funding_usd >= 100_000_000:
                funding_factor = 22.0
            elif funding_usd >= 10_000_000:
                funding_factor = 18.0
            elif funding_usd >= 1_000_000:
                funding_factor = 12.0
            else:
                funding_factor = 6.0
        else:
            funding_factor = 8.0
        breakdown["funding_maturity"] = funding_factor

        if sectors:
            sector_weights = [SECTOR_UAE_WEIGHTS.get(s, 0.5) for s in sectors]
            avg_weight = sum(sector_weights) / len(sector_weights)
            sector_factor = avg_weight * 20.0
        else:
            sector_factor = 10.0
        breakdown["sector_fit"] = round(sector_factor, 1)

        unique_types = set(sig.get("type", "") for sig in signals)
        diversity_factor = min(10.0, len(unique_types) * 2.5)
        breakdown["signal_diversity"] = diversity_factor

        semantic_factor = 5.0
        if self._embedding_agent and description:
            try:
                relevance = self._embedding_agent.relevance_score(description)
                semantic_factor = relevance * 10.0
            except Exception:
                pass
        breakdown["semantic_quality"] = round(semantic_factor, 1)

        total = sum(breakdown.values())
        return min(100.0, max(0.0, total)), breakdown

    def _score_alignment(self, hq_country_code, expansion_targets, sectors, signals, description):
        breakdown = {}
        geo_factor = 0.0
        if hq_country_code == UAE_COUNTRY_CODE:
            geo_factor = 30.0
        elif hq_country_code in GCC_COUNTRY_CODES:
            geo_factor = 22.0
        elif hq_country_code in MENA_COUNTRY_CODES:
            geo_factor = 15.0
        else:
            geo_factor = 5.0
        breakdown["geographic_presence"] = geo_factor

        expansion_factor = 0.0
        expansion_codes = set(expansion_targets)
        if UAE_COUNTRY_CODE in expansion_codes:
            expansion_factor = 25.0
        elif expansion_codes & GCC_COUNTRY_CODES:
            expansion_factor = 18.0
        elif expansion_codes & MENA_COUNTRY_CODES:
            expansion_factor = 12.0
        breakdown["expansion_intent"] = expansion_factor

        if sectors:
            sector_weights = [SECTOR_UAE_WEIGHTS.get(s, 0.5) for s in sectors]
            avg_weight = max(sector_weights)
            sector_factor = avg_weight * 20.0
        else:
            sector_factor = 10.0
        breakdown["strategic_sector_fit"] = round(sector_factor, 1)

        uae_signal_count = 0
        uae_keywords = {"uae", "dubai", "abu dhabi", "difc", "adgm", "vara", "mena", "gcc"}
        for sig in signals:
            text = f"{sig.get('headline', '')} {sig.get('rationale', '')}".lower()
            if any(kw in text for kw in uae_keywords):
                uae_signal_count += 1
        signal_factor = min(15.0, uae_signal_count * 5.0)
        breakdown["uae_signal_relevance"] = signal_factor

        strategy_factor = 5.0
        if self._embedding_agent and description:
            try:
                self._try_load_embeddings()
                desc_emb = self._embedding_agent.encode(description)
                strategy_embs = self._embedding_agent.encode_batch(UAE_STRATEGY_TEXTS)
                sims = strategy_embs @ desc_emb
                max_sim = float(max(sims))
                strategy_factor = max(0.0, min(10.0, max_sim * 12.0))
            except Exception:
                pass
        breakdown["strategic_alignment"] = round(strategy_factor, 1)

        total = sum(breakdown.values())
        return min(100.0, max(0.0, total)), breakdown

    def score(self, signals, sectors, hq_country_code=None, expansion_target_codes=None,
              funding_usd=None, description=None) -> CompanyScores:
        self._try_load_embeddings()
        inv_score, inv_breakdown = self._score_investability(signals, sectors, funding_usd, description)
        ali_score, ali_breakdown = self._score_alignment(
            hq_country_code, expansion_target_codes or [], sectors, signals, description,
        )
        composite = (inv_score + ali_score) / 2
        thesis = self._generate_thesis(sectors, inv_score, ali_score, signals, hq_country_code)
        return CompanyScores(
            investability_score=round(inv_score, 1),
            uae_alignment_score=round(ali_score, 1),
            composite_score=round(composite, 1),
            investability_breakdown=inv_breakdown,
            alignment_breakdown=ali_breakdown,
            thesis_snippet=thesis,
        )

    def score_batch(self, companies: List[Dict]) -> List[CompanyScores]:
        results = []
        for company in companies:
            scores = self.score(
                signals=company.get("signals", []),
                sectors=company.get("sectors", []),
                hq_country_code=company.get("hq_country_code"),
                expansion_target_codes=company.get("expansion_target_codes", []),
                funding_usd=company.get("funding_usd"),
                description=company.get("description"),
            )
            results.append(scores)
        return results

    def _generate_thesis(self, sectors, inv_score, ali_score, signals, hq_code) -> str:
        parts = []
        composite = (inv_score + ali_score) / 2
        if composite >= 80:
            parts.append("High-conviction target.")
        elif composite >= 60:
            parts.append("Strong candidate for Ministry outreach.")
        elif composite >= 40:
            parts.append("Emerging opportunity worth monitoring.")
        else:
            parts.append("Early-stage signal detected.")
        sector_labels = {
            "artificial_intelligence": "AI",
            "cleantech": "cleantech",
            "fintech": "fintech",
            "healthcare": "healthcare",
            "logistics": "logistics",
            "manufacturing": "manufacturing",
            "energy": "energy",
        }
        sector_names = [sector_labels.get(s, s.replace("_", " ")) for s in sectors[:2]]
        if sector_names:
            parts.append(f"Active in {' and '.join(sector_names)}.")
        sig_types = set(s.get("type", "") for s in signals)
        if "funding" in sig_types:
            parts.append("Recent funding activity detected.")
        elif "expansion" in sig_types:
            parts.append("Expansion signals into the region.")
        elif "partnership" in sig_types:
            parts.append("Strategic partnership activity.")
        if hq_code == "AE":
            parts.append("UAE-headquartered.")
        elif hq_code in GCC_COUNTRY_CODES:
            parts.append("GCC-based with regional presence.")
        elif signals and any("uae" in str(s).lower() or "dubai" in str(s).lower() for s in signals):
            parts.append("Shows clear UAE expansion intent.")
        return " ".join(parts)


print("All agents imported successfully!")


## Stage 1: Initialize Agents

In [ ]:
# Initialize all four agents
embedding_agent = EmbeddingAgent()
classifier_agent = ClassifierAgent()
entity_agent = EntityAgent()
scoring_agent = ScoringAgent()

print('Embedding Agent:', 'ML mode' if not embedding_agent._fallback_mode else 'Fallback mode')
print('Classifier Agent: Ready')
print('Entity Agent: Ready')
print('Scoring Agent: Ready')

## Stage 2: Test with Sample Articles

Let's test the pipeline with realistic sample articles before running on live data.

In [ ]:
# Sample articles simulating real RSS feed data
SAMPLE_ARTICLES = [
    {
        'title': 'Tabby raises USD 200M Series D at USD 3.5B valuation',
        'text': 'UAE BNPL fintech Tabby has raised USD 200 million in its Series D round, '
                'led by Wellington Management. The Dubai-based company now has 10M+ active '
                'shoppers across the MENA region and plans to expand to Saudi Arabia.',
        'source': 'Wamda',
        'region': 'MENA',
    },
    {
        'title': 'Bain Capital opens Abu Dhabi office for MENA deals',
        'text': 'US private equity firm Bain Capital has opened a new office in Abu Dhabi, '
                'signaling commitment to regional capital deployment. The firm manages '
                'over USD 180B in assets globally.',
        'source': 'Arabian Business',
        'region': 'GCC',
    },
    {
        'title': 'G42 and Microsoft expand sovereign cloud to five markets',
        'text': 'Abu Dhabi AI company G42 has partnered with Microsoft to deploy sovereign '
                'AI cloud infrastructure across five Middle Eastern markets. The Abu Dhabi '
                'anchor hub will serve as the primary compute node.',
        'source': 'TechCrunch',
        'region': 'Global',
    },
    {
        'title': 'Stripe secures DIFC license, opens Dubai office',
        'text': 'Global payments company Stripe has received regulatory approval from DIFC '
                'to operate in the UAE. The San Francisco-based fintech will open a Dubai '
                'office to serve the MENA payments market.',
        'source': 'Khaleej Times',
        'region': 'UAE',
    },
    {
        'title': 'Manchester United signs new sponsorship deal',
        'text': 'Manchester United football club has signed a new shirt sponsorship deal '
                'worth GBP 60 million per year with a major sportswear brand.',
        'source': 'BBC Sport',
        'region': 'Global',
    },
]

print(f'Loaded {len(SAMPLE_ARTICLES)} sample articles')

## Stage 3: Relevance Scoring (Embedding Agent)

In [ ]:
print('=== Relevance Scores ===')
print(f'{"Article":<55} {"Score":>6} {"Relevant?":>10}')
print('-' * 75)

for article in SAMPLE_ARTICLES:
    text = f"{article['title']} {article['text']}"
    score = embedding_agent.relevance_score(text)
    relevant = 'YES' if score >= 0.2 else 'no'
    print(f'{article["title"][:53]:<55} {score:>5.2f} {relevant:>10}')

## Stage 4: Signal Classification (Classifier Agent)

In [ ]:
print('=== Signal Classification ===')
print(f'{"Article":<40} {"Type":<15} {"Strength":<10} {"Confidence":>10}')
print('-' * 80)

for article in SAMPLE_ARTICLES:
    text = f"{article['title']} {article['text']}"
    result = classifier_agent.classify(text)
    print(f'{article["title"][:38]:<40} {result.signal_type:<15} '
          f'{result.strength:<10} {result.confidence:>9.1%}')

## Stage 5: Entity Extraction (Entity Agent)

In [ ]:
print('=== Entity Extraction ===')
for article in SAMPLE_ARTICLES[:4]:  # Skip the irrelevant one
    entities = entity_agent.extract(article['title'], article['text'])
    print(f'\nArticle: {article["title"][:60]}')
    for e in entities[:2]:
        print(f'  Company: {e.name}')
        print(f'  HQ: {e.headquarters_city or "?"}, {e.headquarters_country or "?"}')
        if e.funding_usd:
            print(f'  Funding: ${e.funding_usd:,.0f}')
        if e.expansion_cities:
            print(f'  Expansion targets: {", ".join(e.expansion_cities)}')

## Stage 6: Scoring (Scoring Agent)

In [ ]:
print('=== Company Scoring ===')
print(f'{"Company":<25} {"Investability":>13} {"UAE Alignment":>13} {"Composite":>10}')
print('-' * 65)

test_companies = [
    {
        'name': 'Tabby',
        'signals': [{'type': 'funding', 'strength': 'high', 'headline': 'Series D raise', 'rationale': 'BNPL growth'}],
        'sectors': ['fintech', 'ecommerce'],
        'hq_country_code': 'AE',
        'expansion_target_codes': ['SA'],
        'funding_usd': 200_000_000,
        'description': 'UAE BNPL fintech with 10M+ shoppers',
    },
    {
        'name': 'Stripe',
        'signals': [{'type': 'expansion', 'strength': 'high', 'headline': 'DIFC license', 'rationale': 'UAE entry'}],
        'sectors': ['fintech'],
        'hq_country_code': 'US',
        'expansion_target_codes': ['AE'],
        'funding_usd': None,
        'description': 'Global payments infrastructure company',
    },
    {
        'name': 'G42',
        'signals': [
            {'type': 'partnership', 'strength': 'high', 'headline': 'Microsoft cloud deal', 'rationale': 'Sovereign AI'},
            {'type': 'funding', 'strength': 'high', 'headline': 'USD 1.5B from Microsoft', 'rationale': 'AI investment'},
        ],
        'sectors': ['artificial_intelligence', 'healthcare'],
        'hq_country_code': 'AE',
        'expansion_target_codes': ['US'],
        'funding_usd': 1_500_000_000,
        'description': 'Abu Dhabi AI and cloud computing company',
    },
]

for company in test_companies:
    scores = scoring_agent.score(**{k: v for k, v in company.items() if k != 'name'})
    print(f'{company["name"]:<25} {scores.investability_score:>12.1f} '
          f'{scores.uae_alignment_score:>12.1f} {scores.composite_score:>9.1f}')
    print(f'  Thesis: {scores.thesis_snippet}')

## Stage 7: Sector Detection (Embedding Agent)

In [ ]:
print('=== Sector Detection ===')
for article in SAMPLE_ARTICLES[:4]:
    text = f"{article['title']} {article['text']}"
    sectors = embedding_agent.top_sectors(text)
    print(f'{article["title"][:50]:<52} -> {", ".join(sectors)}')

## Summary

The pipeline successfully:
- Filters irrelevant articles (sports news scored low)
- Classifies investment signal types with confidence scores
- Extracts company entities, locations, and funding amounts
- Scores companies on investability and UAE alignment
- Detects relevant sectors using semantic embeddings

All of this runs **without any API keys** using open-source models.